In [0]:
# Read Employee Attrition CSV from UC Volume
df = spark.read.csv(
    "`external-catalog-learndatabricks`.default.employee_attrition",
    header=True,
    inferSchema=True
)

print(f"Row count: {df.count()}")
print(f"Columns: {df.columns}")
display(df)

In [0]:
# List Delta Table versions (history)
delta_table_path = "`external-catalog-learndatabricks`.default.employee_attrition"

history_df = spark.sql(f"DESCRIBE HISTORY {delta_table_path}")

display(history_df.select("version", "timestamp", "userId", "userName", "operation", "operationParameters", "isolationLevel", "isBlindAppend"))

In [0]:
# Read Delta Table
df = spark.read.format("delta").table("`external-catalog-learndatabricks`.default.employee_attrition")

print(f"Row count: {df.count()}")
print(f"Columns: {df.columns}")
display(df)

In [0]:
# Insert one dummy record into the Delta table
from pyspark.sql import Row

table_name = "`external-catalog-learndatabricks`.default.employee_attrition"

# Get the table schema to build a matching dummy row
schema = spark.read.format("delta").table(table_name).schema

# Build a dummy record with sensible placeholder values per data type
def dummy_value(field):
    dtype = field.dataType.simpleString()
    if dtype in ("int", "bigint", "smallint", "tinyint"):
        return 0
    elif dtype in ("double", "float", "decimal(10,0)") or dtype.startswith("decimal"):
        return 0.0
    elif dtype == "boolean":
        return False
    elif dtype == "date":
        from datetime import date
        return date(2000, 1, 1)
    elif dtype == "timestamp":
        from datetime import datetime
        return datetime(2000, 1, 1)
    else:  # string and everything else
        return "dummy"

dummy_row = {field.name: dummy_value(field) for field in schema.fields}

# Create a single-row DataFrame and append it to the Delta table
dummy_df = spark.createDataFrame([Row(**dummy_row)], schema=schema)
dummy_df.write.format("delta").mode("append").saveAsTable(table_name)

print("✅ Dummy record inserted successfully.")
display(spark.read.format("delta").table(table_name).orderBy(spark.read.format("delta").table(table_name).columns[0]).limit(5))
